In [1]:
# train.py - Lab 1: Versioned Training Pipeline
# This script trains a Random Forest on the Iris dataset,
# computes metrics, and saves both model and metadata with versioning.

import hashlib  # For computing unique data hash
import json     # For saving metadata
import os
import pickle   # For saving trained model
import numpy as np

from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
"""
# Configuration
This CONFIG dictionary sets the hyperparameters and settings for the Random Forest model and training process.
it includes tree count,depth,random seed,test split size, and a version identifier for tracking the trained model.
"""
CONFIG = {
    "n_estimators": 100,      # Number of trees in the Random Forest
    "max_depth": 5,           # Maximum depth of each tree
    "random_state": 42,       # Seed for reproducibility
    "test_size": 0.2,         # Fraction of data used for testing
    "model_version": "v1.0.0" # Version of the model
}

# -------------------------------
# Compute SHA-256 hash of dataset
# -------------------------------
def compute_data_hash(X, y):
    data_bytes = X.tobytes() + y.tobytes()  # Convert arrays to bytes
    return hashlib.sha256(data_bytes).hexdigest()

# Load and split Iris dataset
def load_and_split_data():
    iris = load_iris()
    X, y = iris.data, iris.target

    # Split dataset into training and testing
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=CONFIG["test_size"],
        random_state=CONFIG["random_state"]
    )

    return X_train, X_test, y_train, y_test, X, y
# train the Random Forest model
"""
This function initializes a Random Forest classifier using the hyperparameters from CONFIG,trains 
it on the provided X_train and y_train data,
and returns the trained model ready for evaluation or prediction.
"""

def train_model(X_train, y_train):
    model = RandomForestClassifier(
        n_estimators=CONFIG["n_estimators"],
        max_depth=CONFIG["max_depth"],
        random_state=CONFIG["random_state"]
    )
    model.fit(X_train, y_train)  # Train model on training data
    return model

# -------------------------------
# Evaluate the model
# -------------------------------
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "f1_score": f1_score(y_test, y_pred, average='weighted')
    }
    return metrics

# -------------------------------
# Run the entire training pipeline
# -------------------------------
def run_training():
    print("[INFO] Starting training pipeline")

    # Load and split dataset
    X_train, X_test, y_train, y_test, X, y = load_and_split_data()
    print("[INFO] Train:", len(X_train), "Test:", len(X_test))

    # Compute data hash for reproducibility
    data_hash = compute_data_hash(X, y)
    print("[INFO] Data hash:", data_hash)

    # Train model
    model = train_model(X_train, y_train)

    # Evaluate model
    metrics = evaluate_model(model, X_test, y_test)
    print("[INFO] Metrics:", metrics)

    # Save trained model
    with open("model.pkl", "wb") as f:
        pickle.dump(model, f)
    print("[INFO] Model saved as 'model.pkl'")

    # Save metadata including version, metrics, config, and data hash
    metadata = {
        "model_version": CONFIG["model_version"],
        "metrics": metrics,
        "data_hash": data_hash,
        "config": CONFIG
    }
    with open("metadata.json", "w") as f:
        json.dump(metadata, f, indent=4)
    print("[INFO] Metadata saved as 'metadata.json'")

    print("[SUCCESS] Accuracy:", metrics.get("accuracy", 0))
    return model, metrics, data_hash

# -------------------------------
# Main entry point
# -------------------------------
if __name__ == '__main__':
    run_training()



[INFO] Starting training pipeline
[INFO] Train: 120 Test: 30
[INFO] Data hash: aa06b8008ceba42efc654be0f83fdafc786239c9e8f13146044d078f5aab8f23
[INFO] Metrics: {'accuracy': 1.0, 'f1_score': 1.0}
[INFO] Model saved as 'model.pkl'
[INFO] Metadata saved as 'metadata.json'
[SUCCESS] Accuracy: 1.0



validate.py - Lab 2: Model Validation Gates



In [2]:
# validate.py - Lab 2: Model Validation Gates

import numpy as np
from sklearn.metrics import accuracy_score, f1_score, recall_score
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

#threshold is used to check whether the model failed or not
THRESHOLDS = {
    "min_accuracy": 0.85,
    "min_f1": 0.80,
    "regression_tolerance": 0.02,
    "min_per_class_recall": 0.70,
    "expected_feature_count": 4,
}
PROD_BASELINE = {"accuracy": 0.88, "f1": 0.87}  # Previous production model performance


def load_test_data():
    iris = load_iris()  # load dataset
    X, y = iris.data, iris.target
    _, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42)  # Split data into tran and test
# this will Return only test data
    return X_test, y_test


def gate_schema_validation(X_test):
    """
    if n_features checks if number of features are equal to expected thresholds ,
    if it not matches it returns false
    it checks  there are missing values (NaN) in the dataset ,if there are no missing values returns true
    if there are missing values returns false
    """
    # X_test.shape gives the dimensions of the data and shape gives number of columns (features)
    n_features = X_test.shape[1]

    if n_features != THRESHOLDS["expected_feature_count"]:  # Check feature count
        return False, f"Expected {THRESHOLDS['expected_feature_count']} features, got {n_features}"

    if np.isnan(X_test).any():  # Check for missing values
        return False, "NaN values detected"

    return True, "Schema valid"  # Passed schema validation
"""
This function checks how well the model performs by calculating accuracy and F1 score using the test data.
it will compares these values with  thresholds to ensure the model meets minimum quality standards.
 if either metric is too low, it fails the validation,otherwise it passes successfully.
"""

def gate_performance(model, X_test, y_test):

    preds = model.predict(X_test)  # Make predictions

    acc = accuracy_score(y_test, preds)  # Calculate accuracy
    f1 = f1_score(y_test, preds, average="weighted")  # Calculate F1 score

    metrics = {"accuracy": acc, "f1_score": f1}  # Store metrics

    if acc < THRESHOLDS["min_accuracy"]:  # Check accuracy threshold
        return False, metrics, f"Accuracy {acc:.4f} < {THRESHOLDS['min_accuracy']}"

    if f1 < THRESHOLDS["min_f1"]:  # Check F1 threshold
        return False, metrics, f"F1 {f1:.4f} < {THRESHOLDS['min_f1']}"

    return True, metrics, "Performance OK"  # Passed performance check

"""
it will get new metrics
this will check the new model accuracy with previous
"""

def gate_regression(new_metrics):

    new_accuracy = new_metrics["accuracy"]  # Get new model accuracy

    # Check if accuracy dropped more than allowed tolerance
    if new_accuracy < (PROD_BASELINE["accuracy"] - THRESHOLDS["regression_tolerance"]):
        return False, f"Regression detected: {new_accuracy:.4f} dropped below allowed baseline"

    return True, "No regression"  # No regression issue


def gate_fairness(model, X_test, y_test):
    preds = model.predict(X_test)  # Make predictions

    per_class = recall_score(y_test, preds, average=None)  # Recall for each class

    # Find classes with recall below threshold
    failing = [i for i, r in enumerate(per_class) if r < THRESHOLDS["min_per_class_recall"]]

    if failing:  # If any class fails
        return False, dict(enumerate(per_class)), f"Classes {failing} below threshold"

    return True, dict(enumerate(per_class)), "Fairness OK"  # All classes passed

"""
Gates are used to control and verify model quality before deployment.
each gate acts like a checkpoint that ensures the model meets specific conditions
such as correct data format, good performance, no regression, and fairness.
If any gate fails, the process stops, preventing a poor or unsafe model from being deployed.
"""
"""
this function runs all validation checks schema, performance, regression, and fairness one by one on the model.
If any gate fails, it immediately stops and returns the failure reason.
"""
def run_all_gates(model=None):

    print('=' * 50)
    print('VALIDATION PIPELINE')
    print('=' * 50)

    from sklearn.ensemble import RandomForestClassifier
    X_test, y_test = load_test_data()  # Load test dataset

    if model is None:  # If no model provided, train a default one
        iris = load_iris()
        X_tr, _, y_tr, _ = train_test_split(iris.data, iris.target, test_size=0.2, random_state=42)
        model = RandomForestClassifier(n_estimators=100, random_state=42)
        model.fit(X_tr, y_tr)

    # Gate 1
    passed, msg = gate_schema_validation(X_test)  # Run schema validation
    print(f"[GATE 1] Schema: {'PASS' if passed else 'FAIL'} - {msg}")
    if not passed:
        return {"status": "FAIL", "failed_gate": "schema", "reason": msg}

    # Gate 2
    passed, metrics, msg = gate_performance(model, X_test, y_test)  # Run performance check
    print(f"[GATE 2] Performance: {'PASS' if passed else 'FAIL'} - {msg}")
    if not passed:
        return {"status": "FAIL", "failed_gate": "performance", "reason": msg, "metrics": metrics}

    # Gate 3
    passed, msg = gate_regression(metrics)  # Run regression check
    print(f"[GATE 3] Regression: {'PASS' if passed else 'FAIL'} - {msg}")
    if not passed:
        return {"status": "FAIL", "failed_gate": "regression", "reason": msg, "metrics": metrics}

    # Gate 4
    passed, per_class, msg = gate_fairness(model, X_test, y_test)  # Run fairness check
    print(f"[GATE 4] Fairness: {'PASS' if passed else 'FAIL'} - {msg}")
    if not passed:
        return {"status": "FAIL", "failed_gate": "fairness", "reason": msg, "per_class_recall": per_class}

    return {"status": "PASS", "metrics": metrics}  # All gates passed


if __name__ == '__main__':
    result = run_all_gates()  # Execute pipeline
    print("\nFINAL:", result["status"])

VALIDATION PIPELINE
[GATE 1] Schema: PASS - Schema valid
[GATE 2] Performance: PASS - Performance OK
[GATE 3] Regression: PASS - No regression
[GATE 4] Fairness: PASS - Fairness OK

FINAL: PASS


Lab 3: drift_detect.py 

In [3]:
# drift_detect.py - Lab 3: Statistical Drift Detection
# PSI > 0.1 = slight | PSI > 0.2 = severe (trigger retrain!)
"""
Drift detection is the process of identifying whether the data in production has changed,
when compared to the data used during training.
"""

import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

PSI_SLIGHT = 0.1
PSI_SEVERE = 0.2
"""
it loads the  iris dataset from sklearn.it splits the dataset into train and test parts using train_test_split
"""
def get_reference_data():
    iris = load_iris()
    X_train, _, y_train, _ = train_test_split(iris.data, iris.target, test_size=0.2, random_state=42)
    return X_train, y_train


def get_production_data(drift_magnitude=0.8):
    np.random.seed(99)
    iris = load_iris()
    X = iris.data.copy()
    X[:, 0] += drift_magnitude * 1.5
    X[:, 2] += drift_magnitude * 0.8
    X += np.random.normal(0, drift_magnitude * 0.3, X.shape)
    return X[:100], iris.target[:100]

"""
PSI - Population Stability Index measures how much a feature’s distribution has changed between training and production data.
PSI helps detect data drift.
"""
def compute_psi(reference, production, n_bins=10):

    # Create histogram bins based on reference data range
    # Why: ensures consistent binning for both reference and production datasets
    bins = np.linspace(reference.min(), reference.max(), n_bins + 1)

    # Compute histogram counts for reference and production data
    # What it does: counts how many samples fall into each bin
    ref_counts, _ = np.histogram(reference, bins=bins)
    prod_counts, _ = np.histogram(production, bins=bins)

    # Convert counts to percentages
    # Why: PSI formula requires proportions instead of raw counts
    ref_pct = ref_counts / np.sum(ref_counts)
    prod_pct = prod_counts / np.sum(prod_counts)

    # Clip percentages to avoid division by zero or log(0)
    # What it does: prevents numerical instability
    ref_pct = np.clip(ref_pct, 1e-6, 1)
    prod_pct = np.clip(prod_pct, 1e-6, 1)

    # Compute PSI using the standard formula
    # What it does: measures distribution drift between reference and production
    psi = np.sum((prod_pct - ref_pct) * np.log(prod_pct / ref_pct))

    return psi

"""
KL Divergence measures how different the production data distribution is from the reference distribution.
It’s another way to detect data drift
"""
def compute_kl_divergence(reference, production, n_bins=10):
    # Create histogram bins based on reference data
    # ensures consistent binning for both datasets
    bins = np.linspace(reference.min(), reference.max(), n_bins + 1)

    # Compute histogram counts for reference and production
    #  it converts continuous data into discrete probability distributions
    ref_counts, _ = np.histogram(reference, bins=bins)
    prod_counts, _ = np.histogram(production, bins=bins)

    # Convert counts to probabilities
    ref_prob = ref_counts / np.sum(ref_counts)
    prod_prob = prod_counts / np.sum(prod_counts)

    # Clip probabilities to avoid division by zero or log(0)
    ref_prob = np.clip(ref_prob, 1e-6, 1)
    prod_prob = np.clip(prod_prob, 1e-6, 1)

    # Compute KL divergence
    kl = np.sum(prod_prob * np.log(prod_prob / ref_prob))

    return kl


def detect_feature_drift(X_ref, X_prod, feature_names=None):
    # Generate default feature names if not provided
    # Why: ensures each feature has a human-readable identifier for reporting
    if feature_names is None:
        feature_names = ["f" + str(i) for i in range(X_ref.shape[1])]

    results = []
    """
    This loop goes feature by feature and calculates drift metrics PSI & KL
    to determine how much each feature has changed in production data compared to reference data.
    """
    # Iterate over each feature column
    for i, fname in enumerate(feature_names):
        # Extract reference and production feature columns
        ref_col = X_ref[:, i]
        prod_col = X_prod[:, i]

        # Compute PSI for the feature
        # it measures distribution shift between reference and production
        psi = compute_psi(ref_col, prod_col)

        # Compute KL divergence for the feature
        #  quantifies divergence of production distribution from reference
        kl_div = compute_kl_divergence(ref_col, prod_col)

        # Determine severity based on PSI thresholds
        #categorizes drift as none, slight, or severe for alerting
        if psi > PSI_SEVERE:
            severity = "severe"
            alert = True  # Trigger retraining or alert for severe drift
        elif psi > PSI_SLIGHT:
            severity = "slight"
            alert = False  # Minor drift, log for monitoring
        else:
            severity = "none"
            alert = False  # No significant drift detected

        # Append feature drift info to results
        # What it does: collects all metrics and severity info per feature in a structured format
        results.append({
            "feature": fname,
            "psi": float(psi),       # Ensure JSON-serializable
            "kl_div": float(kl_div), # Ensure JSON-serializable
            "severity": severity,
            "alert": alert
        })

    return results


def check_prediction_drift(model, X_ref, X_prod):
    # ### YOUR CODE ###
    # Generate predictions for reference and production datasets
    # Why: comparing predicted class distributions helps detect concept drift
    y_ref_pred = model.predict(X_ref)
    y_prod_pred = model.predict(X_prod)

    # Compute class proportions for reference predictions
    # What it does: normalizes counts to percentages for fair comparison
    ref_counts = np.bincount(y_ref_pred, minlength=len(np.unique(y_ref_pred)))
    ref_prop = ref_counts / np.sum(ref_counts)

    # Compute class proportions for production predictions
    prod_counts = np.bincount(y_prod_pred, minlength=len(np.unique(y_ref_pred)))
    prod_prop = prod_counts / np.sum(prod_counts)

    # Initialize drift flag
    drift_detected = False
    changes = {}

    # Compare proportions for each class
    # Why: large differences may indicate model predictions are drifting
    for i, (r, p) in enumerate(zip(ref_prop, prod_prop)):
        change = abs(p - r)  # Absolute difference
        changes[f"class_{i}"] = float(change)  # Ensure JSON-serializable

        # Flag drift if change exceeds threshold
        # What it does: identifies classes where model behavior differs significantly
        if change > 0.15:
            drift_detected = True

    return drift_detected, changes


def generate_drift_report(feature_results, pred_drift, pred_changes):
 
    # Initialize lists to track drifted features by severity
    # What it does: helps summarize which features caused drift
    severe_features = []
    slight_features = []

    for feat in feature_results:
        if feat["severity"] == "severe":
            severe_features.append(feat["feature"])
        elif feat["severity"] == "slight":
            slight_features.append(feat["feature"])

    # Determine overall status based on feature drift and prediction drift
    # it ensures the most critical issue dictates alert color
    if severe_features or pred_drift:
        overall_status = "RED"      # severe issue, immediate attention needed
    elif slight_features:
        overall_status = "YELLOW"   # minor drift, monitor closely
    else:
        overall_status = "GREEN"    # no significant drift detected

    # Prepare recommendation text
    # What it does: guides next actions based on detected drift
    if overall_status == "RED":
        recommendation = "Immediate investigation or retraining recommended."
    elif overall_status == "YELLOW":
        recommendation = "Monitor features and model performance."
    else:
        recommendation = "No action needed."

    # Construct structured report
    report = {
        "overall_status": overall_status,
        "drifted_features": severe_features + slight_features,
        "recommendation": recommendation
    }

    return report


def run_drift_detection():
    print('=' * 50 + '\nDRIFT DETECTION\n' + '=' * 50)
    X_ref, y_ref = get_reference_data()
    X_prod, y_prod = get_production_data(drift_magnitude=0.8)
    names = ["sepal_length","sepal_width","petal_length","petal_width"]
    feature_results = detect_feature_drift(X_ref, X_prod, names)
    print('\nFeature Drift:')
    for r in feature_results:
        sev = r.get("severity","?")
        icon = "RED" if sev=="severe" else ("YLW" if sev=="slight" else " OK")
        print("  [" + icon + "] " + r["feature"] + ": PSI=" + str(round(r.get("psi",0),4)))
    from sklearn.ensemble import RandomForestClassifier
    model = RandomForestClassifier(n_estimators=50, random_state=42)
    model.fit(X_ref, y_ref)
    pred_drift, pred_changes = check_prediction_drift(model, X_ref, X_prod)
    print('Prediction drift:', pred_drift)
    report = generate_drift_report(feature_results, pred_drift, pred_changes)
    print("STATUS:", report.get("overall_status","UNKNOWN"))
    return report


if __name__ == '__main__':
    run_drift_detection()

DRIFT DETECTION

Feature Drift:
  [RED] sepal_length: PSI=2.5382
  [YLW] sepal_width: PSI=0.1744
  [RED] petal_length: PSI=4.2984
  [RED] petal_width: PSI=2.4715
Prediction drift: True
STATUS: RED


LAB 04
ci_pipeline.yml



ci_pipeline.yml
name: ML Training CI Pipeline

# ---------------------------------------------
# Trigger the CI pipeline
# ---------------------------------------------
on:
  push:
    # Run pipeline only if changes occur in these directories/files
    paths:
      - 'src/**'       # source code
      - 'configs/**'   # config files
      - 'data/**'      # datasets

  schedule:
    - cron: '0 2 * * *'

  # Allow manual trigger
  workflow_dispatch:

# ---------------------------------------------
# Environment variables
# ---------------------------------------------
env:
  PYTHON_VERSION: '3.10'           # Python version for the workflow
  MODEL_VERSION: ${{ github.sha }} # Use git commit SHA as model version

# =============================================
# Job 1: Train the model
# =============================================
jobs:
  train:
    name: Train Model
    runs-on: ubuntu-latest  # Run on latest Ubuntu VM
    steps:
      # Step 1: Checkout repository code
      - uses: actions/checkout@v4

      # Step 2: Setup Python environment
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ env.PYTHON_VERSION }}
          cache: 'pip'  # cache pip packages for faster installs

      # Step 3: Install dependencies
      - name: Install dependencies
        run: |
          pip install -r requirements.txt

      # Step 4: Run training script
      - name: Run training
        run: |
          python train.py
        env:
          MODEL_VERSION: ${{ env.MODEL_VERSION }}

      # Step 5: Upload trained model as artifact
      - name: Upload model artifact
        uses: actions/upload-artifact@v4
        with:
          name: model-${{ github.sha }}
          path: model.pkl
          retention-days: 30  # keep artifact for 30 days

# =============================================
# Job 2: Validate the model
# =============================================
  validate:
    name: Validate Model
    runs-on: ubuntu-latest
    needs: [train]  # Wait for 'train' job to finish
    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with:
          python-version: ${{ env.PYTHON_VERSION }}
          cache: 'pip'

      # Install dependencies again
      - run: pip install -r requirements.txt

      # Download the model artifact from training
      - uses: actions/download-artifact@v4
        with:
          name: model-${{ github.sha }}

      # Run the validation gates
      - name: Run validation gates
        run: |
          python validate.py

# =============================================
# Job 3: Register the model
# =============================================
  register:
    name: Register Model
    runs-on: ubuntu-latest
    needs: [validate]                  # Wait for 'validate' job
    if: github.ref == 'refs/heads/main'  # Only run on main branch
    steps:
      - uses: actions/checkout@v4

      # Install dependencies
      - run: pip install -r requirements.txt

      # Example of registration step (can be replaced with actual ML registry code)
      - run: python -c "print('Registered: iris_classifier @ Staging')"



===========================
      Output
==============================
 
    mlops-lab $ 
$ yaml-lint ci_pipeline.yml
[CHECK] Push path triggers: OK
[CHECK] Schedule cron: OK - nightly at 02:00 UTC
[CHECK] MODEL_VERSION = github.sha: OK
[CHECK] pip install: OK
[CHECK] validate needs train: OK

In [ ]:
Lab 5: Canary Deployment


# cd_deploy.yml - Lab 5: Canary Deployment
# Runs after CI pipeline succeeds on main

name: ML Model CD - Canary Deployment 

"""
Canary deployment is a technique to release a new version of a model or software to a small subset of users first, 
instead of releasing to everyone immediately.
"""
"""
this event triggers a workflow when another workflow completes.
Unlike push or pull_request, this is specifically for continuous deployment pipelines, which depend on the CI pipeline finishing first.
Here it is used for canary deployment after the CI pipeline succeeds.
"""

on:
  # Trigger when 'ML Training CI Pipeline' workflow completes
  workflow_run:
    workflows: ["ML Training CI Pipeline"]  # CI workflow name
    types: [completed]                      # Trigger only when workflow completes
    branches: [main]                        # Only for main branch

jobs:

  canary_deploy:
    name: Deploy Canary (10% traffic)
    runs-on: ubuntu-latest
    # Only run if CI workflow SUCCEEDED
    if: github.event.workflow_run.conclusion == 'success'

    outputs:
      canary_healthy: ${{ steps.health_check.outputs.healthy }}
      model_version: ${{ steps.deploy.outputs.version }}
"""
deploy the new ML model to 10% of users canary traffic,
 wait for metrics, and check if the canary is healthy.
"""
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.10'
      - run: pip install -r requirements.txt

      - name: Deploy canary at 10% traffic
        id: deploy
        run: |
          python -c "
          version = 'v-abc1234'
          print('[DEPLOY] Canary 10% - version:', version)
          # Pass version to outputs for other jobs
          import os
          with open(os.environ['GITHUB_OUTPUT'], 'a') as f:
              f.write('version=' + version + chr(10))
          "

      - name: Wait for metrics to stabilize
        run: |
          echo 'Waiting for canary metrics...'
          sleep 60  # wait 60 seconds for traffic to accumulate metrics

      - name: Check canary health
        id: health_check
        run: |
          python -c "
          import random
          import os
          random.seed(42)
          accuracy = round(random.uniform(0.87, 0.94), 4)
          latency  = round(random.uniform(45, 110), 1)
          print('[METRICS] accuracy:', accuracy, 'latency:', latency, 'ms')
          healthy = accuracy >= 0.85 and latency <= 200
          print('[METRICS] healthy:', healthy)
          # Write healthy flag to GITHUB_OUTPUT
          with open(os.environ['GITHUB_OUTPUT'], 'a') as f:
              f.write('healthy=' + str(healthy).lower() + chr(10))
          "

  full_rollout:
    name: Promote to 100% Traffic
    runs-on: ubuntu-latest
    needs: canary_deploy
    # Only run if canary is healthy
    if: needs.canary_deploy.outputs.canary_healthy == 'true'
    steps:
      - run: |
          echo '[PROMOTE] 100% traffic -> new model'
          echo '[PROMOTE] Stage: Staging -> Production'
"""
Only runs if the canary health check failed.
Prevents bad model from reaching 100% of production traffic.
"""

  rollback:
    name: Auto Rollback
    runs-on: ubuntu-latest
    needs: canary_deploy
    # Only run if canary failed
    if: needs.canary_deploy.outputs.canary_healthy != 'true'
    steps:
      - run: |
          echo '[ROLLBACK] Canary FAILED - reverting to stable model'
          echo '[ALERT] On-call team notified'
=================
Output
==================
[CHECK] Checks conclusion == 'success': OK
[CHECK] Wait/sleep step: OK
[CHECK] full_rollout if canary_healthy==true: OK
[CHECK] rollback if canary_healthy!=true: OK
[SIMULATE] Running canary deployment...
[DEPLOY] Canary 10% traffic - version: v-abc1234
[METRICS] accuracy: 0.9142 latency: 78.3ms
[METRICS] healthy: True -> PROMOTING to 100%


monitor.py - Lab 6: Production Monitor 

In [4]:
# monitor.py - Lab 6: Production Monitor + Auto-Retraining
# Runs every hour. Closes the CI/CD feedback loop.

import json
import numpy as np
from datetime import datetime, timedelta

# Configuration for monitoring thresholds
MONITOR_CONFIG = {
    "accuracy_drop_threshold": 0.05,       # alert if accuracy drops more than 5%
    "max_retrains_per_day": 2,             # maximum retrains allowed per day
    "min_hours_between_retrains": 4,
    "baseline_accuracy": 0.90,            
}

# Simulated production state
class ProductionState:
    def __init__(self):
        np.random.seed(42)
        # Recent predictions (simulated)
        self.recent_predictions = np.random.randint(0, 3, 500)
        # Simulate true labels with some noise
        noise = np.random.binomial(1, 0.12, 500).astype(bool)
        self.recent_true_labels = self.recent_predictions.copy()
        self.recent_true_labels[noise] = (self.recent_true_labels[noise] + 1) % 3
        # Production and reference feature means & stds (simulated)
        self.prod_means = np.array([5.9, 3.2, 4.8, 1.8])
        self.ref_means  = np.array([5.1, 3.0, 3.8, 1.2])
        self.ref_stds   = np.array([0.8, 0.4, 1.8, 0.8])
        self.retrain_log = []  # Track retraining timestamps

# Initialize production state
state = ProductionState()

# -------------------------------
# Rolling accuracy check
# -------------------------------
"""
this function calculates the models rolling accuracy by comparing predictions to true labels,
computes the drop from baseline accuracy, and flags an alert if the drop exceeds a threshold.
it also assigns a severity level: none, warning, critical depending on how big the drop is.
"""
def check_rolling_accuracy(predictions, true_labels):
    correct = (predictions == true_labels).sum()
    accuracy = correct / len(predictions)
    drop = MONITOR_CONFIG["baseline_accuracy"] - accuracy
    alert = drop > MONITOR_CONFIG["accuracy_drop_threshold"]
    threshold = MONITOR_CONFIG["accuracy_drop_threshold"]
    severity = "critical" if drop > 2*threshold else ("warning" if alert else "none")
    return {
        "accuracy": accuracy,
        "drop_from_baseline": drop,
        "alert": alert,
        "severity": severity
    }

# -------------------------------
# Feature drift check (simplified)
# -------------------------------
"""
this function checks for feature drift by comparing production feature means to  training means.
it computes a z-score style drift score for each feature: |prod_mean - ref_mean| / ref_std.
If the score exceeds 2.0, the feature is flagged as drifted, and results for all features are returned.
"""
def check_feature_drift_simplified(prod_means, ref_means, ref_stds):
    names = ["sepal_length","sepal_width","petal_length","petal_width"]
    results = []
    for i, name in enumerate(names):
        score = abs(prod_means[i] - ref_means[i]) / ref_stds[i]  # z-score style
        results.append({
            "feature": name,
            "drift_score": score,
            "drifted": score > 2.0
        })
    return results

# -------------------------------
# Circuit breaker for retraining
# -------------------------------
def check_circuit_breaker(retrain_log):
    now = datetime.utcnow()
    # Count retrains in last 24 hours
    last_24h = [t for t in retrain_log if now - t < timedelta(hours=24)]
    if len(last_24h) >= MONITOR_CONFIG["max_retrains_per_day"]:
        return False, "Max daily retrains reached"
    # Check minimum hours since last retrain
    if last_24h:
        hours_since = (now - max(last_24h)).seconds / 3600
        if hours_since < MONITOR_CONFIG["min_hours_between_retrains"]:
            return False, f"Too soon: {hours_since:.1f}h since last retrain"
    return True, ""

# -------------------------------
# Trigger retraining workflow
# -------------------------------
def trigger_retraining(reason, severity, metrics):
    payload = {
        "ref": "main",
        "inputs": {
            "trigger_reason": reason,
            "severity": severity,
            "metrics": metrics
        }
    }
    print("[RETRAIN] Payload to CI/CD pipeline:")
    print(json.dumps(payload, indent=2))
    triggered = True
    run_url = "https://github.com/myorg/ml-pipeline/actions/runs/99999"
    return triggered, run_url

# -------------------------------
# Generate alert for monitoring dashboard
# -------------------------------
def generate_alert(issues, metrics):
    severity = "RED" if any(i.get("type")=="accuracy_drop" for i in issues) else "YELLOW"
    alert = {
        "timestamp": datetime.utcnow().isoformat(),
        "severity": severity,
        "issues": issues,
        "recommended_action": "Retrain model or investigate feature drift"
    }
    return alert

# -------------------------------
# Monitoring cycle
# -------------------------------
"""
Rolling Accuracy Check : compares recent predictions vs true labels.
computes accuracy, drop from baseline, and alert severity.
Feature Drift Check : compares current feature stats (means) vs reference stats.
                      calculates z-score style drift for each feature.
                      flags features as drifted if z-score > 2.0.
"""
def run_monitoring_cycle():
    print('MONITORING CYCLE -', datetime.utcnow().strftime('%Y-%m-%d %H:%M UTC'))
    issues = []
    overall_status = 'GREEN'

    # 1. Check accuracy
    acc = check_rolling_accuracy(state.recent_predictions, state.recent_true_labels)
    print('[ACCURACY]', acc.get('accuracy'), '| drop:', acc.get('drop_from_baseline'))
    if acc.get('alert'):
        issues.append({'type': 'accuracy_drop', 'details': acc})
        overall_status = 'RED' if acc.get('severity') == 'critical' else 'YELLOW'

    # 2. Check feature drift
    drift = check_feature_drift_simplified(state.prod_means, state.ref_means, state.ref_stds)
    print('[DRIFT]')
    for r in drift:
        print(' ', r['feature'], 'score=' + str(round(r.get('drift_score',0),3)), 'DRIFTED' if r.get('drifted') else 'OK')
        if r.get('drifted'):
            issues.append({'type': 'feature_drift', 'feature': r['feature']})
            if overall_status == 'GREEN':
                overall_status = 'YELLOW'

    print('[STATUS]', overall_status, '| Issues:', len(issues))

    # 3. Retraining if issues exist
    if issues:
        can_retrain, block_reason = check_circuit_breaker(state.retrain_log)
        if can_retrain:
            triggered, url = trigger_retraining(
                reason=issues[0]['type'],
                severity=overall_status,
                metrics={'accuracy': acc.get('accuracy')}
            )
            if triggered:
                state.retrain_log.append(datetime.utcnow())
                print('[RETRAIN] Triggered:', url)
        else:
            print('[CIRCUIT BREAKER] Blocked:', block_reason)
            alert = generate_alert(issues, acc)
            print('[ALERT]', json.dumps(alert, indent=2))

    return overall_status, issues

# -------------------------------
# Run monitoring when script is executed
# -------------------------------
if __name__ == '__main__':
    status, issues = run_monitoring_cycle()
    print('Final status:', status)


MONITORING CYCLE - 2026-03-20 12:23 UTC
[ACCURACY] 0.89 | drop: 0.010000000000000009
[DRIFT]
  sepal_length score=1.0 OK
  sepal_width score=0.5 OK
  petal_length score=0.556 OK
  petal_width score=0.75 OK
[STATUS] GREEN | Issues: 0
Final status: GREEN


AI tools are used during this assignement to get  guidance on implementing different stages of the ML pipeline